In [ ]:
!pip install transformers datasets evaluate torch tqdm

In [ ]:
import torch
import time
import numpy as np
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)
import evaluate

# Config
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 512
DOC_STRIDE = 128
BATCH_SIZE = 8

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
metric = evaluate.load("squad")

In [ ]:
train_dataset = load_dataset("squad", split="train[:3%]")
val_dataset = load_dataset("squad", split="validation[:3%]")

print(train_dataset[0])

In [ ]:
def prepare_baseline(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=0,
        return_offsets_mapping=True,
        padding="max_length"
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(tokenized["offset_mapping"]):
        answers = examples["answers"][i]
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        sequence_ids = tokenized.sequence_ids(i)
        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized

In [ ]:
def prepare_sliding(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        sequence_ids = tokenized.sequence_ids(i)
        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized

In [ ]:
train_baseline = train_dataset.map(prepare_baseline, batched=True, remove_columns=train_dataset.column_names)
train_sliding = train_dataset.map(prepare_sliding, batched=True, remove_columns=train_dataset.column_names)

In [ ]:
def train_model(dataset, name):
    print(f"\nRunning: {name}")

    model = BertForQuestionAnswering.from_pretrained(MODEL_NAME)

    args = TrainingArguments(
        output_dir=f"./{name}",
        learning_rate=2e-5,
        per_device_train_batch_size=BATCH_SIZE,
        num_train_epochs=1,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DefaultDataCollator(),
    )

    start = time.time()
    trainer.train()
    total_time = time.time() - start

    params = sum(p.numel() for p in model.parameters())

    print(f"{name} Time: {total_time:.2f} sec")
    print(f"{name} Params: {params}")

    return total_time, params

In [ ]:
base_time, base_params = train_model(train_baseline, "Baseline")

In [ ]:
imp_time, imp_params = train_model(train_sliding, "Sliding_Window")

In [ ]:
print("\nFINAL RESULTS")
print("Baseline Time:", base_time)
print("Improved Time:", imp_time)
print("Parameters:", base_params)